In [13]:

#################################################################################################################################################################################################################################################################################
#CONEXION AL ORACLE

import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM cmcpp10", con=connection2)





In [14]:
base2

,cpscod,cpsdes,uniacpcod,tipexacod,areaexacod,tipprucod,cpsmueflg,cpsind,estprogpocod,estproseccod,...,cpsusumodcod,cpsmodfec,cpsobs,cpsdescor,cpsdurmin,catresexaauxflg,tipresexaauxcod,cpsateterflg,cpsexaproflg,cpspattipoflg
0,73220,"IMAGENES POR RESONANCIA MAGNETICA, EXTREMIDAD ...",5,1,1,2,0,\r\n,2,01,...,06722213,2010-06-02 11:01:40,None,None,None,0,None,0,None,None
1,73221,"IMAGENES POR RESONANCIA MAGNETICA, CUALQUIER A...",5,1,1,2,0,.,2,01,...,EPASTOR,2013-09-30 08:53:17,,,None,0,None,0,None,None
2,73223,"IMAGENES POR RESONANCIA MAGNETICA, CUALQUIER A...",5,1,1,2,0,\r\n,2,01,...,06722213,2010-06-02 11:02:52,None,None,None,0,None,0,None,None
3,73225,"ANGIOGRAFIA POR RESONANCIA MAGNETICA, EXTREMIN...",5,1,1,None,0,,2,01,...,EPASTOR,2022-04-25 09:17:26,,ANGIORESONANCIA EXTREMINDAD SUPERIOR,None,0,None,0,0,0
4,73510,"CADERA, UNILATERAL; COMPLETO, MINIMO DE DOS VI...",5,1,1,None,0,None,2,01,...,None,None,None,RX CADERA UNILATERAL,None,0,None,0,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12838,91299.01,ENDOSCOPIA DE MAGNIFICACION DE TUBO DIGESTIVO ...,2,None,None,None,0,None,2,04,...,None,None,None,None,None,0,None,None,None,None
12839,91299.02,ENDOSCOPIA DE MAGNIFICACION DE TUBO DIGESTIVO ...,2,None,None,None,0,None,2,04,...,None,None,None,None,None,0,None,None,None,None
12840,91299.03,DISECCION ENDOSCOPICA DE LA SUBMUCOSA DEL TUBO...,2,None,None,None,0,None,2,04,...,None,None,None,None,None,0,None,None,None,None
12841,91299.04,DISECCION ENDOSCOPICA DE LA SUBMUCOSA DEL TUBO...,2,None,None,None,0,None,2,04,...,None,None,None,None,None,0,None,None,None,None


In [15]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()



#connection3.execute('CREATE TEMPORARY TABLE tmp_cmcpp10 ()')
base2.to_sql(name='tmp_cmcpp10', con=engine3, if_exists='replace', index=False)
#df.to_sql(name='temp_table', con=engine, if_exists='replace', index=False, method='multi', chunksize=1000, temporary=True)



#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL cmcpp10 FALSO CON LO OBTENIDO DEL ORACLE

query="""

ALTER TABLE tmp_cmcpp10 
ALTER COLUMN cpscod TYPE character (10)  ,
ALTER COLUMN cpsdes TYPE character (500)  ,
ALTER COLUMN uniacpcod TYPE character (1)  ,
ALTER COLUMN tipexacod TYPE character (1)  ,
ALTER COLUMN areaexacod TYPE character (1)  ,
ALTER COLUMN tipprucod TYPE character (1)  ,
ALTER COLUMN cpsmueflg TYPE character (1)  ,
ALTER COLUMN cpsind TYPE character (200)  ,
ALTER COLUMN estprogpocod TYPE character (1)  ,
ALTER COLUMN estproseccod TYPE character (2)  ,
ALTER COLUMN estprossecod TYPE character (2)  ,
ALTER COLUMN estpronivcod TYPE character (2)  ,
ALTER COLUMN cpsestregcod TYPE character (1)  ,
ALTER COLUMN estpidcod TYPE character (2)  ,
ALTER COLUMN grdcmccod TYPE character (1)  ,
ALTER COLUMN cpsusucrecod TYPE character (10)  ,
ALTER COLUMN cpscrefec TYPE date,
ALTER COLUMN cpsusumodcod TYPE character (10)  ,
ALTER COLUMN cpsmodfec TYPE date,
ALTER COLUMN cpsobs TYPE character  (200)  ,
ALTER COLUMN cpsdescor TYPE character  (80)  ,
ALTER COLUMN cpsdurmin TYPE numeric(3,0) USING (CASE WHEN cpsdurmin='' THEN NULL ELSE cpsdurmin::numeric END),
ALTER COLUMN catresexaauxflg TYPE character (1)  ,
ALTER COLUMN tipresexaauxcod TYPE character (3)  ,
ALTER COLUMN cpsateterflg TYPE character (1)  ,
ALTER COLUMN cpsexaproflg TYPE character (1)  ,
ALTER COLUMN cpspattipoflg TYPE character (1)  ;


UPDATE cmcpp10 
SET 
cpsdes=t.cpsdes,
uniacpcod=t.uniacpcod,
tipexacod=t.tipexacod,
areaexacod=t.areaexacod,
tipprucod=t.tipprucod,
cpsmueflg=t.cpsmueflg,
cpsind=t.cpsind,
estprogpocod=t.estprogpocod,
estproseccod=t.estproseccod,
estprossecod=t.estprossecod,
estpronivcod=t.estpronivcod,
cpsestregcod=t.cpsestregcod,
estpidcod=t.estpidcod,
grdcmccod=t.grdcmccod,
cpsusucrecod=t.cpsusucrecod,
cpscrefec=t.cpscrefec,
cpsusumodcod=t.cpsusumodcod,
cpsmodfec=t.cpsmodfec,
cpsobs=t.cpsobs,
cpsdescor=t.cpsdescor,
cpsdurmin=t.cpsdurmin,
catresexaauxflg=t.catresexaauxflg,
tipresexaauxcod=t.tipresexaauxcod,
cpsateterflg=t.cpsateterflg,
cpsexaproflg=t.cpsexaproflg,
cpspattipoflg=t.cpspattipoflg                

FROM tmp_cmcpp10 t 
WHERE cmcpp10.cpscod = t.cpscod;

INSERT INTO cmcpp10 (cpscod, cpsdes, uniacpcod, tipexacod, areaexacod, tipprucod,
cpsmueflg, cpsind, estprogpocod, estproseccod, estprossecod,
estpronivcod, cpsestregcod, estpidcod, grdcmccod,
cpsusucrecod, cpscrefec, cpsusumodcod, cpsmodfec, cpsobs,
cpsdescor, cpsdurmin, catresexaauxflg, tipresexaauxcod,
cpsateterflg, cpsexaproflg, cpspattipoflg) 

SELECT cpscod, cpsdes, uniacpcod, tipexacod, areaexacod, tipprucod,
cpsmueflg, cpsind, estprogpocod, estproseccod, estprossecod,
estpronivcod, cpsestregcod, estpidcod, grdcmccod,
cpsusucrecod, cpscrefec, cpsusumodcod, cpsmodfec, cpsobs,
cpsdescor, cpsdurmin, catresexaauxflg, tipresexaauxcod,
cpsateterflg, cpsexaproflg, cpspattipoflg
FROM tmp_cmcpp10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM cmcpp10 
    WHERE cmcpp10.cpscod = tmp_cmcpp10.cpscod
);


"""
#SE OMITIO CPDURMIN POR SER TOTALMENTE NULO 

c1= text(query)
connection3.execute(c1)



#BORRAMOS LAS TABLAS
base2 = pd.read_sql_query(f"SELECT * FROM cmcpp10", con=connection3)
query2="""
DROP TABLE tmp_cmcpp10;
"""

c2= text(query2)
connection3.execute(c2)



In [16]:
base2.columns

Index(['cpsmodfec', 'cpscrefec', 'cpsdurmin', 'cpspattipoflg', 'cpsexaproflg',
       'cpsateterflg', 'tipresexaauxcod', 'catresexaauxflg', 'cpsdescor',
       'cpsobs', 'cpsusumodcod', 'cpsusucrecod', 'grdcmccod', 'estpidcod',
       'cpsestregcod', 'estpronivcod', 'estprossecod', 'estproseccod',
       'estprogpocod', 'cpsind', 'cpsmueflg', 'tipprucod', 'areaexacod',
       'tipexacod', 'uniacpcod', 'cpsdes', 'cpscod'],
      dtype='object')

In [17]:


#################################################################################################################################################################################################################################################################################


#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="INDICADORES_ESSALUD"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine4 = create_engine(cadena4)
connection4 = engine4.connect()


base2.to_sql(name='tmp_cmcpp10', con=engine4, if_exists='replace', index=False)



#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS

query="""
UPDATE dim_cps 
SET des_cps      = t.cpsdes


FROM tmp_cmcpp10 t 
WHERE TRIM(dim_cps.cod_cps) = TRIM(t.cpscod);

INSERT INTO dim_cps (cod_cps, des_cps) 
SELECT cpscod,cpsdes
FROM tmp_cmcpp10  
WHERE NOT EXISTS (
    SELECT 1 
    FROM dim_cps 
    WHERE TRIM(dim_cps.cod_cps) = TRIM(tmp_cmcpp10.cpscod)
);


"""
#DROP TABLE tmp_cmcpp10;
c1= text(query)
connection4.execute(c1)


In [18]:
connection2.close()
connection3.close()
connection4.close()